In [ ]:
from math import ceil
import numpy as np
import pandas as pd
import statsmodels.stats.api as sms
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

# 1. Load & clean data
df = pd.read_csv('ab_data.csv')
session_counts = df['user_id'].value_counts()
multi_users = session_counts[session_counts > 1].index
df_clean = df[~df['user_id'].isin(multi_users)].copy()


In [ ]:
# 2. Sample size calculation
p1, p2 = 0.13, 0.15
mde_target = p2 - p1
effect_size = sms.proportion_effectsize(p1, p2)

required_n = sms.NormalIndPower().solve_power(
    effect_size=effect_size, power=0.80, alpha=0.05, ratio=1.0
)
required_n = ceil(required_n)

In [ ]:
# 3. Random sampling
control_sample = df_clean[df_clean['group'] == 'control'].sample(
    n=required_n, random_state=22
)
treatment_sample = df_clean[df_clean['group'] == 'treatment'].sample(
    n=required_n, random_state=22
)
ab_test = pd.concat([control_sample, treatment_sample], axis=0).reset_index(
    drop=True
)

In [ ]:
# 4. Summary stats
conversion_rates = ab_test.groupby('group')['converted'].agg(
    ['count', 'sum', 'mean', 'std']
)
conversion_rates.columns = [
    'sample_size',
    'conversions',
    'conv_rate',
    'std_dev',
]
print(conversion_rates, '\n')

p_A = conversion_rates.loc['control', 'conv_rate']
p_B = conversion_rates.loc['treatment', 'conv_rate']
delta = p_B - p_A

In [ ]:
# 5. Hypothesis testing (Z-test)
control_res = ab_test[ab_test['group'] == 'control']['converted']
treatment_res = ab_test[ab_test['group'] == 'treatment']['converted']

successes = [treatment_res.sum(), control_res.sum()]
nobs = [treatment_res.count(), control_res.count()]

z_stat, p_val = proportions_ztest(count=successes, nobs=nobs)
(lower_treat, lower_con), (upper_treat, upper_con) = proportion_confint(
    successes, nobs, alpha=0.05
)

print(f"Z-statistic : {z_stat:.4f}")
print(f"p-value     : {p_val:.4f}")
print(f"CI Control  : [{lower_con:.2%}, {upper_con:.2%}]")
print(f"CI Treatment: [{lower_treat:.2%}, {upper_treat:.2%}]\n")

In [ ]:
# 6. Evaluation
print(f"Lift (p_B - p_A): {delta:+.2%}")
if p_val < 0.05:
    if delta < 0:
        print("Decision: Reject H0. Variant B performed worse. Keep Control.")
    elif delta < mde_target:
        print(
            "Decision: Reject H0. Improved, but below MDE target. Evaluate ROI."
        )
    else:
        print(
            "Decision: Reject H0. Fully optimal. Exceeded MDE target. Roll out 100%."
        )
else:
    print(
        "Decision: Fail to reject H0. No statistically significant difference. Keep Control."
    )